# Compare screened and unscreened pattern search

**Question:** Does the learned search step change exact-engine cost and selected decisions on a common problem?

Level: 3. Prerequisites: earlier lessons in this topic. Reading time: 30 minutes.



## Objectives

The paired-search lesson measures exact evaluation cost and selected-decision agreement on the same magnetite problem at weight 0.5. Endpoint statuses and independent physical deltas qualify whether the two reported solutions actually agree, rather than inferring agreement from rounded recovery.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Import the pinned analysis tools once and resolve inputs relative to the notebook folder.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. Inputs are the authored `iron_magnetite_fine` case in the production catalog and `data/derived/contract/operating_contract.json`. No measured plant dataset or economic model is used. Screening reads the committed process_screen.json, process_surrogate.json and companion model files under models/. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The screened and unscreened production optimizers solve the same one-dimensional magnetite problem at weight 0.5. Candidate cost is counted by exact circuit evaluations, while the learned search step only prioritizes proposals. Both reported endpoints are independently audited when their states can be solved. A separate table calculates screened-minus-unscreened physical decisions and divides each difference by its contract range. The explicitly stated 1e-8 tolerance tests numerical decision agreement, not optimality. Agreement additionally requires feasible endpoints and available audits; physical metric deltas and cost differences remain separate evidence.

## Comparison

The direct reference is the same production pattern search with the learned proposal step disabled. Both calls retain case, base, weight 0.5, contract and stopping settings. The additional unscreened record inside the screened call supplies a fresh-cache consistency reference. Exact work, decision tolerance and independent metric differences are measured on the actual pair rather than inherited from a stored optimum.

Construct an identical magnetite optimization problem for two named search configurations: progressive-barrier pattern search with and without the learned proposal step. The common nominal state determines objective scales and the operating contract determines admissible grind targets. This comparison restricts dimensionality to keep the complete production search affordable. It does not shorten the declared stopping rule or substitute an analytic stand-in for the process engine.

In [ ]:
# Use the authored magnetite circuit as a bounded one-dimensional search example.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.optimization import optimize, Problem
from pipeline.methods.pattern_search import pattern_search
from pipeline.engine.constants import constant

case = CASE_BY_ID["iron_magnetite_fine"]
contract = json.loads(
    (repo / "data/derived/contract/operating_contract.json").read_text(encoding="utf-8")
)
base = simulate(case.ore, case.plant, case.nominal)
print("Base recovery (%):", round(base.metrics["recovery_pct"], 3))
print("Decision bounds:", contract["cases"][case.id]["inputs"]["target_p80_um"])


Execute the unscreened reference first and keep its result intact. Its candidate count establishes the cost baseline, while its endpoint supplies physical outcomes and slacks. Record all start stop reasons so a feasible budget-limited result is not confused with mesh convergence. Exact engine evaluations are the comparison unit; the lesson does not use elapsed seconds, which would mix algorithmic work with host load, caches and numerical-library startup.

In [ ]:
# Run the production multistart search without a learned screen, with equal weights on normalized
# recovered metal and energy.
plain = optimize(case, case.nominal, contract, weight=0.5, path=False, screen=None)
print("Unscreened status:", plain["status"])
print("Unscreened evaluations:", plain["evaluations"])
print("Stopping reasons:", [r["stop"] for r in plain["starts"]])
plain_end = plain.get("optimum") or plain.get("least_violating") or plain["base"]
display(pd.Series(plain_end["slacks"], name="Constraint slack").to_frame())


Enable the committed screen without changing case, base, weight or bounds. The proposal step uses a learned model to prioritize an unvisited point, but the exact engine must still judge it. A poor proposal can add work before the normal poll continues. The returned independent unscreened pair uses a fresh cache, providing an internal paired reference as well as the explicit reference call already performed.

In [ ]:
# Load the committed float64 screen and run the same production optimizer with screening enabled.
from pipeline.methods.screen import Screen

screen = Screen(repo / "models")
screened = optimize(case, case.nominal, contract, weight=0.5, path=False, screen=screen)
screened_end = screened.get("optimum") or screened.get("least_violating") or screened["base"]
print("Screened status:", screened["status"])
print("Screened evaluations:", screened["evaluations"])
print("Paired unscreened evaluations:", screened.get("without_screen", {}).get("evaluations"))
proposals = [
    row for run in screened["starts"] for row in run.get("screen", {}).get("proposals", [])
]
proposal_table = pd.DataFrame(
    proposals,
    columns=screened.get(
        "proposal_columns",
        (
            "iteration",
            "surrogate_recovery_pct",
            "engine_recovery_pct",
            "surrogate_objective",
            "engine_objective",
            "runner_up_margin",
            "improved",
        ),
    ),
)
display(proposal_table.head(8))
print("Proposals:", len(proposals))
print("Additional evaluations:", screened["evaluations"] - plain["evaluations"])


The screened candidate is audited with the exact circuit independently of the learned score. Its refusal or physical response is retained for the paired comparison, so a numerical match cannot certify two infeasible or unaudited endpoints.

In [ ]:
# Preserve unavailable screened audits instead of inventing terminal physical metrics.
from pipeline.engine.model import InfeasibleState

screened_point = case.nominal.with_values(**screened_end["decisions"])
screened_check = None
if screened["status"] != "base_refused":
    try:
        screened_check = simulate(case.ore, case.plant, screened_point)
    except InfeasibleState as refusal:
        print("Screened audit refusal:", refusal.code)
print("Screened audit available:", screened_check is not None)
if screened_check is not None:
    print("Screened engine recovery (%):", round(screened_check.metrics["recovery_pct"], 3))
print("Screened constraint slacks:", screened_end["slacks"])


The explicit unscreened reference endpoint is re-evaluated outside the optimizer cache. Its audit either supplies physical metrics for the later matched-delta table or remains unavailable with a refusal message. Those audit values qualify physical agreement and are not counted as candidate evaluations or as evidence that the search is globally optimal.

In [ ]:
# Audit the reported unscreened endpoint only when the engine can define that state.
from pipeline.engine.model import InfeasibleState

plain_point = case.nominal.with_values(**plain_end["decisions"])
plain_check = None
if plain["status"] != "base_refused":
    try:
        plain_check = simulate(case.ore, case.plant, plain_point)
    except InfeasibleState as refusal:
        print("Unscreened audit refusal:", refusal.code)
print("Unscreened audit available:", plain_check is not None)
if plain_check is not None:
    print("Unscreened engine recovery (%):", round(plain_check.metrics["recovery_pct"], 3))
    print(
        "Unscreened engine energy (kWh/t):",
        round(plain_check.metrics["specific_energy_total_kwh_t"], 3),
    )


Plot the reference incumbent objective and mesh. The curve records the selected start, so it should not be read as a chronological merge of all starts. A plateau does not mean that the engine was idle: rejected candidates and feasibility improvements still cost evaluations. This is the mechanism through which adding a prediction step can change total cost without changing the final feasible operating point.

In [ ]:
# Plot the recorded incumbent objective and mesh history of the winning unscreened start.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 8})
history = pd.DataFrame(
    plain.get("trace", []), columns=["Outcome", "Mesh", "Barrier", "Objective", "Evaluations"]
)
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
axes[0].plot(history["Evaluations"], history["Objective"], marker=".")
axes[0].set(xlabel="Exact calls (n)", ylabel="Objective (1)", title="Objective progress")
axes[1].step(range(len(history)), history["Mesh"], where="post")
axes[1].set(xlabel="Iteration (n)", ylabel="Mesh (1)", title="Search mesh")
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


Interpret the late iterations with the reported mesh stop and feasibility tolerance. The production algorithm's convergence criterion is an explicit computational rule, not an empirical proof that the selected grind is the best possible plant decision. The circuit has authored mineral parameters and no independently calibrated uncertainty margin. Preserve those limits when comparing a method's search efficiency with its physical usefulness.

Place screened and unscreened exact counts side by side. Both bars include a complete production multistart solve on the same one-dimensional contract. A higher screened bar is a negative result for the proposed cost benefit on this example. A lower bar would be a local saving, not a claim that the learned model makes every circuit search faster. Inspect the proposal table before attributing the direction to prediction accuracy alone.

In [ ]:
# Compare the completed runs in the unit that the experiment actually measures, keeping all
# outcomes in view..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 8})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.bar(["Unscreened", "Screened"], [plain["evaluations"], screened["evaluations"]])
ax.set(xlabel="Search method", ylabel="Exact calls (n)", title="Fresh paired engine cost")
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.87)


The cost comparison is paired on inputs and settings but covers one authored circuit. It measures exact objective work and does not include model-training cost, audit solves or wall-clock inference overhead. A matching terminal recovery therefore supports only physical agreement within the displayed precision. The next evidence lesson examines individual proposal errors and compares this fresh bounded run with the current committed cohort, identified separately by engine version, contract digest and file SHA-256. Pair counts and signed costs come from the records actually read.

The independent unscreened reference returned inside the screened call is useful for checking that the explicit baseline used the same problem. Compare its engine count with the separate plain call and inspect the common decision bounds and start inventory. A difference would require explaining settings or state, not averaging the two counts. The proposal table records surrogate and engine objectives separately because a prediction can rank well and still fail the exact improvement test. A trusted prediction interval constrains model use but is not an oracle for optimization benefit.

The two search configurations are summarized before decision agreement is calculated. Their exact calls, terminal statuses and physical slacks establish whether the subsequent normalized decision test compares feasible audited outcomes. Start-level stops distinguish a budget-limited endpoint from a mesh-converged reference rather than treating both as one success category.

In [ ]:
# Report each run before comparing cost or preference.
runs = {"Unscreened": plain, "Screened": screened}
endpoint_rows = []
for label, run in runs.items():
    end = run.get("optimum") or run.get("least_violating") or run["base"]
    physical = run["status"] != "base_refused" and bool(end["values"])
    endpoint_rows.append(
        {
            "Run": label,
            "Status": run["status"],
            "Feasible": end["feasible"],
            "Converged": run.get("converged"),
            "Stops": ", ".join((r["stop"] for r in run["starts"])),
            "Decisions": end["decisions"],
            "Slacks": end["slacks"],
            "Exact calls": run["evaluations"],
            "Recovered metal (t/h)": end["recovered_tph"] if physical else None,
            "Recovery (%)": end["recovery_pct"] if physical else None,
            "Specific energy (kWh/t)": end["values"].get("energy_kwh_t") if physical else None,
        }
    )
endpoints = pd.DataFrame(endpoint_rows)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(endpoints)


Compute screened-minus-unscreened decision differences in physical units and normalized by each contract range. The displayed tolerance 1e-8 is a declared numerical comparison rule, not a proof of global optimality. The next calculation applies physical feasibility and audit qualifications separately.

In [ ]:
# Measure actual decision disagreement independently of cost.
decision_tolerance = 1e-08
decision_rows = []
for name, (lo, hi) in plain["bounds"].items():
    left, right = (plain_end["decisions"][name], screened_end["decisions"][name])
    delta = right - left
    decision_rows.append(
        {
            "Decision": name,
            "Unit": contract["cases"][case.id]["inputs"][name]["unit"],
            "Unscreened": left,
            "Screened": right,
            "Physical delta": delta,
            "Normalized delta (1)": delta / (hi - lo),
            "Tolerance (1)": decision_tolerance,
            "Within tolerance": abs(delta / (hi - lo)) <= decision_tolerance,
        }
    )
decision_comparison = pd.DataFrame(decision_rows)


The physical qualification applies the computed tolerance result only when both endpoints are feasible and both audits exist. The metric deltas then compare recovery and energy on the same engine basis; unavailable audits remain missing values rather than zero differences.

In [ ]:
# Qualify numerical decision agreement with physical audits.
both_audited = plain_check is not None and screened_check is not None
decision_agreement = bool(
    plain_end["feasible"]
    and screened_end["feasible"]
    and both_audited
    and decision_comparison["Within tolerance"].all()
)
display(decision_comparison)
print("Feasible audited decision agreement:", decision_agreement)
physical_deltas = {"Recovery delta (pp)": None, "Energy delta (kWh/t)": None}
if both_audited:
    physical_deltas["Recovery delta (pp)"] = (
        screened_check.metrics["recovery_pct"] - plain_check.metrics["recovery_pct"]
    )
    physical_deltas["Energy delta (kWh/t)"] = (
        screened_check.metrics["specific_energy_total_kwh_t"]
        - plain_check.metrics["specific_energy_total_kwh_t"]
    )
display(pd.Series(physical_deltas, name="Screened minus unscreened").to_frame())


## Results

**Answer:** Exact costs are <!-- result:Screened count -->0<!-- /result --> with screening and <!-- result:Unscreened count -->0<!-- /result --> without it; screened-minus-unscreened excess is <!-- result:Additional count -->0<!-- /result -->. Feasible audited decision agreement is <!-- result:Audited decision agreement (1=yes) -->0<!-- /result --> (1=yes), with largest normalized delta <!-- result:Largest normalized decision delta (1) -->0<!-- /result --> against tolerance 1e-8. Cost direction follows the signed excess, while the separate metric-delta table tests physical agreement rather than assuming it from a rounded recovery.

Collect paired exact counts and status-qualified normalized decision agreement from the separately calculated comparisons.

In [ ]:
# Collect status-qualified search outcomes without inventing missing physical metrics.
answer_values = {
    "Unscreened count": plain["evaluations"],
    "Reported target P80 (µm)": plain_end["decisions"]["target_p80_um"],
    "Selected feasible (1=yes)": int(plain_end["feasible"]),
    "Convergence reported (1=yes)": int(plain.get("converged") is True),
}
answer_values.update(
    {
        "Screened count": screened["evaluations"],
        "Additional count": screened["evaluations"] - plain["evaluations"],
        "Proposal count": len(proposals),
        "Screened feasible (1=yes)": int(screened_end["feasible"]),
        "Audited decision agreement (1=yes)": int(decision_agreement),
        "Largest normalized decision delta (1)": float(
            decision_comparison["Normalized delta (1)"].abs().max()
        ),
    }
)
answer_values = {label: round(float(number), 8) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

Paired cost and numerical decision agreement are separately measured. A lower exact-call count would not certify better physics, and a matched recovery rounded for display would not establish equal decisions. The tables expose the underlying endpoint and normalized deltas.

## References

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662
- https://doi.org/10.1007/BF01197708
- https://doi.org/10.1017/CBO9780511790423